In [1]:
"""
TreeWalker Strategy Test Notebook

Investigating two potential issues:
1. When does get_best_action return None (no conclusive best action)?
2. Does tighten_value_estimate_gap ever change the best action?
"""
import numpy as np
import pandas as pd
import os
import sys

from blackjack.actions import PlayerAction, DealerAction
from blackjack.blackjack_round import BJRound, BJStage
from blackjack_cpp import ProbabilisticRankShoe, RandomSampler, load_combo_data, TreeWalker
from blackjack.rules import BJRules
import random
from tqdm import tqdm
import time


In [2]:
# Load combo data for the TreeWalker
current_dir = os.getcwd()
load_combo_data(os.path.join(current_dir, "../combinations/s16_new/"), 11)

# Setup rules
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=True,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3 / 2,
    surrender_payout=1 / 2,
    insurance_payout=2 / 1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)


## Investigation 1: When does get_best_action return None?

`get_best_action()` returns `None` when there's no conclusively-best action - i.e., when multiple actions could potentially be optimal based on the floor/ceiling value estimates.

This happens when `meaningful_actions_.size() != 1` in the `DecisionNode`.

Let's run many simulations and track when this occurs.


In [3]:
from blackjack.blackjack_round import BJRound, BJStage
from blackjack.actions import DealerAction, PlayerAction


def build_tree_walker_if_needed(tree_walker, bj_round, shoe, bet_unit):
    """Build TreeWalker on first action needed."""
    if tree_walker is not None:
        return tree_walker
    if not (bj_round.need_player_action() or bj_round.need_dealer_action()):
        return None
    return TreeWalker.build_initial(
        bj_round.player_hands[0].cards,
        bj_round.dealer_hand[0],
        bj_round.rules.to_cpp(),
        shoe.get_rank_count(),
        bet_unit=bet_unit,
        initial_cards_burned=True
    )


def handle_player_action(bj_round, tree_walker, shoe, tracking):
    """Handle player action, track ambiguity, return chosen action."""
    tracking['total_decisions'] += 1
    
    best_action = tree_walker.get_best_action()
    best_actions = tree_walker.get_best_actions()
    dealer_upcard = bj_round.get_dealer_hand()[0]
    
    if best_action is None:
        tracking['ambiguous_decisions'] += 1
        tracking['ambiguous_cases'].append({
            'player_cards': tuple(bj_round.player_hands[0].cards),
            'dealer_card': dealer_upcard,
            'best_actions': [(a, e.ev, e.ev_min, e.ev_max) for a, e in best_actions],
            'shoe_counts': dict(shoe.get_rank_count()),
            'cards_remaining': sum(shoe.get_rank_count().values()),
        })
        best_action = PlayerAction[best_actions[0][0]]
    else:
        best_action = PlayerAction[best_action]
    
    bj_round.take_action(best_action)
    tree_walker.take_player_action(best_action.value)
    return best_action


def handle_dealer_bj_check(bj_round, tree_walker, shoe):
    """Handle dealer blackjack check. Returns the dealer's second card."""
    dealer_upcard = bj_round.get_dealer_hand()[0]
    second_card_for_bj = 10 if dealer_upcard == 11 else 11
    dealer_second_card = shoe.sample_and_burn_rank()
    
    if dealer_second_card == second_card_for_bj:
        action = DealerAction.CONFIRM_BLACKJACK
    else:
        action = DealerAction.CONFIRM_NO_BLACKJACK
        shoe.lock_dealer_card_not(second_card_for_bj)
    
    bj_round.take_action(action)
    tree_walker.take_dealer_action(action.value)
    return dealer_second_card


def handle_card_draw(bj_round, tree_walker, shoe, stage, dealer_second_card):
    """Handle card draw for player or dealer. Returns updated dealer_second_card."""
    if stage == BJStage.DEALER_CARD and dealer_second_card is not None:
        next_card = dealer_second_card
        dealer_second_card = None
    else:
        possible_cards = bj_round.get_possible_next_card_ranks()
        next_card = shoe.sample_and_burn_rank(possible_cards)
    
    if stage == BJStage.PLAYER_CARD and tree_walker is not None:
        tree_walker.take_card(next_card)
    bj_round.take_card(next_card)
    return dealer_second_card


def play_round_and_check_ambiguity(rules, shoe, bet_unit=100):
    """Play a single round and check for ambiguous decisions."""
    shoe.unlock_dealer_card()
    bj_round = BJRound(rules)
    bj_round.start_round(bet_unit=bet_unit)
    
    tracking = {
        'total_decisions': 0,
        'ambiguous_decisions': 0,
        'ambiguous_cases': [],
    }
    
    tree_walker = None
    dealer_second_card = None
    
    try:
        while bj_round.get_stage() != BJStage.ROUND_OVER:
            tree_walker = build_tree_walker_if_needed(tree_walker, bj_round, shoe, bet_unit)
            stage = bj_round.get_stage()
            possible_actions = bj_round.get_available_actions()
            
            if len(possible_actions) > 0:
                # Action phase
                if bj_round.need_player_action():
                    handle_player_action(bj_round, tree_walker, shoe, tracking)
                elif stage == BJStage.DEALER_CHECK_BJ:
                    dealer_second_card = handle_dealer_bj_check(bj_round, tree_walker, shoe)
                else:
                    raise RuntimeError(f"Unexpected action stage: {stage}")
            else:
                # Card drawing phase (including initial deal)
                dealer_second_card = handle_card_draw(bj_round, tree_walker, shoe, stage, dealer_second_card)
        
        return True, tracking
    except Exception:
        return False, tracking


def exhaust_shoe(rules, shoe, min_cards, on_round_complete):
    """Play rounds until shoe is nearly exhausted."""
    while sum(shoe.get_rank_count().values()) >= min_cards:
        success, tracking = play_round_and_check_ambiguity(rules, shoe)
        if not success:
            break
        on_round_complete(tracking)


def test_best_action_ambiguity_exhaustive(n_shoes=500, min_cards=20, seed=42):
    """Test ambiguity by exhausting shoes - creates edge cases with skewed distributions."""
    random.seed(seed)
    
    totals = {'decisions': 0, 'ambiguous': 0}
    ambiguous_cases = []
    
    def accumulate(tracking):
        totals['decisions'] += tracking['total_decisions']
        totals['ambiguous'] += tracking['ambiguous_decisions']
        ambiguous_cases.extend(tracking['ambiguous_cases'])
    
    for _ in tqdm(range(n_shoes)):
        shoe = ProbabilisticRankShoe(n_decks=6, seed=random.randrange(10000000))
        exhaust_shoe(rules, shoe, min_cards, accumulate)
    
    return {
        'total_decisions': totals['decisions'],
        'ambiguous_decisions': totals['ambiguous'],
        'ambiguity_rate': totals['ambiguous'] / totals['decisions'] if totals['decisions'] > 0 else 0,
        'ambiguous_cases': ambiguous_cases,
        'shoes_played': n_shoes,
    }


# Run the exhaustive test
print("Testing ambiguity by exhausting shoes (edge cases with skewed distributions)...")
seed = time.time() % 1000000
print(f"Seed: {seed}")
results = test_best_action_ambiguity_exhaustive(n_shoes=10000, min_cards=20, seed=seed)
print(f"\nShoes played: {results['shoes_played']}")
print(f"Total decisions: {results['total_decisions']}")
print(f"Ambiguous decisions: {results['ambiguous_decisions']}")
print(f"Ambiguity rate: {results['ambiguity_rate']:.4%}")


Testing ambiguity by exhausting shoes (edge cases with skewed distributions)...
Seed: 204917.17839837074


100%|██████████| 10000/10000 [16:02<00:00, 10.39it/s]


Shoes played: 10000
Total decisions: 153587
Ambiguous decisions: 0
Ambiguity rate: 0.0000%


In [4]:
# Display some ambiguous cases with shoe information
print("Sample ambiguous cases (with shoe state):")
print("=" * 100)
for i, case in enumerate(results['ambiguous_cases'][:15]):
    player = case['player_cards']
    dealer = case['dealer_card']
    cards_left = case['cards_remaining']
    shoe_counts = case['shoe_counts']
    
    print(f"\nCase {i+1}: Player {player} vs Dealer {dealer}")
    print(f"  Cards remaining: {cards_left}")
    print(f"  Shoe: {shoe_counts}")
    print("  Best actions (action, ev, ev_min, ev_max):")
    for action, ev, ev_min, ev_max in case['best_actions']:
        gap = ev_max - ev_min
        print(f"    {action:20s}: EV={ev:8.4f}  [{ev_min:8.4f}, {ev_max:8.4f}]  gap={gap:.4f}")


Sample ambiguous cases (with shoe state):


## Investigation 2: Does tighten_value_estimate_gap change the best action?

We want to know if running `tighten_value_estimate_gap` with aggressive parameters (1e-5 or 1e-6) ever changes:
1. The "best action" returned by `get_best_action()` (or resolves None to an action)
2. The first action in the list returned by `get_best_actions()`

For this test, we'll:
1. Create a TreeWalker
2. Record the initial best action / first of best actions
3. Call tighten_value_estimate_gap with aggressive gap (1e-5, 1e-6)
4. Compare if the action changed


In [5]:
def test_tighten_on_single_decision(tree_walker, player_cards, dealer_card, shoe_counts, gap):
    """Test if tightening changes the best action for a single decision."""
    initial_best_action = tree_walker.get_best_action()
    initial_best_actions = tree_walker.get_best_actions()
    initial_first_action = initial_best_actions[0][0] if initial_best_actions else None
    
    tree_walker.tighten_value_estimate_gap(gap)
    
    final_best_action = tree_walker.get_best_action()
    final_best_actions = tree_walker.get_best_actions()
    final_first_action = final_best_actions[0][0] if final_best_actions else None
    
    return {
        'best_action_changed': initial_best_action != final_best_action,
        'first_action_changed': initial_first_action != final_first_action,
        'best_action_resolved': initial_best_action is None and final_best_action is not None,
        'player_cards': player_cards,
        'dealer_card': dealer_card,
        'shoe_counts': shoe_counts,
        'initial_first_action': initial_first_action,
        'final_first_action': final_first_action,
        'initial_best_actions': [(a, e.ev, e.ev_min, e.ev_max) for a, e in initial_best_actions],
        'final_best_actions': [(a, e.ev, e.ev_min, e.ev_max) for a, e in final_best_actions],
    }


def handle_player_action_with_tighten(bj_round, tree_walker, shoe, gap, stats):
    """Handle player action with tightening test. Updates stats dict."""
    stats['total_decisions'] += 1
    
    result = test_tighten_on_single_decision(
        tree_walker,
        tuple(bj_round.player_hands[0].cards),
        bj_round.get_dealer_hand()[0],
        dict(shoe.get_rank_count()),
        gap
    )
    
    if result['best_action_changed']:
        stats['best_action_changed'] += 1
    if result['best_action_resolved']:
        stats['best_action_resolved'] += 1
    if result['first_action_changed']:
        stats['first_action_changed'] += 1
        stats['changed_cases'].append(result)
    
    # Use the final best action to continue
    best_actions = tree_walker.get_best_actions()
    best_action = PlayerAction[best_actions[0][0]]
    bj_round.take_action(best_action)
    tree_walker.take_player_action(best_action.value)


def play_round_with_tighten_test(rules, shoe, gap, stats, bet_unit=100):
    """Play a round while testing tighten effect on each decision."""
    shoe.unlock_dealer_card()
    bj_round = BJRound(rules)
    bj_round.start_round(bet_unit=bet_unit)
    
    tree_walker = None
    dealer_second_card = None
    
    try:
        while bj_round.get_stage() != BJStage.ROUND_OVER:
            tree_walker = build_tree_walker_if_needed(tree_walker, bj_round, shoe, bet_unit)
            stage = bj_round.get_stage()
            possible_actions = bj_round.get_available_actions()
            
            if len(possible_actions) > 0:
                # Action phase
                if bj_round.need_player_action():
                    handle_player_action_with_tighten(bj_round, tree_walker, shoe, gap, stats)
                elif stage == BJStage.DEALER_CHECK_BJ:
                    dealer_second_card = handle_dealer_bj_check(bj_round, tree_walker, shoe)
                else:
                    raise RuntimeError(f"Unexpected action stage: {stage}")
            else:
                # Card drawing phase (including initial deal)
                dealer_second_card = handle_card_draw(bj_round, tree_walker, shoe, stage, dealer_second_card)
        return True
    except Exception:
        return False


def test_tighten_changes_action_exhaustive(n_shoes=3000, min_cards=20, gap=1e-5, seed=42):
    """Test if tighten_value_estimate_gap ever changes the best action."""
    random.seed(seed)
    
    stats = {
        'total_decisions': 0,
        'best_action_changed': 0,
        'best_action_resolved': 0,
        'first_action_changed': 0,
        'changed_cases': [],
    }
    
    for _ in tqdm(range(n_shoes)):
        shoe = ProbabilisticRankShoe(n_decks=6, seed=random.randrange(10000000))
        
        while sum(shoe.get_rank_count().values()) >= min_cards:
            if not play_round_with_tighten_test(rules, shoe, gap, stats):
                break
    
    stats['gap'] = gap
    return stats



In [6]:
# Test with 1e-6 gap (more aggressive)
print("\nTesting with gap = 1e-6 (more aggressive)")
seed = time.time() % 1000000
print(f"Seed: {seed}")
results_1e6 = test_tighten_changes_action_exhaustive(n_shoes=3000, min_cards=20, gap=1e-6, seed=seed)
print(f"Total decisions: {results_1e6['total_decisions']}")
print(f"Best action changed (get_best_action): {results_1e6['best_action_changed']} ({results_1e6['best_action_changed']/results_1e6['total_decisions']:.4%})")
print(f"  - Of which None->action resolved: {results_1e6['best_action_resolved']}")
print(f"First action changed (get_best_actions[0]): {results_1e6['first_action_changed']} ({results_1e6['first_action_changed']/results_1e6['total_decisions']:.4%})")



Testing with gap = 1e-6 (more aggressive)
Seed: 205879.6403245926


100%|██████████| 3000/3000 [25:29<00:00,  1.96it/s] 

Total decisions: 50151
Best action changed (get_best_action): 0 (0.0000%)
  - Of which None->action resolved: 0
First action changed (get_best_actions[0]): 0 (0.0000%)


In [8]:
# Display cases where first action changed (with shoe state)
def show_changed_cases(results, max_cases=5):
    print(f"\nCases where first action changed (gap={results['gap']}):")
    print("=" * 100)
    
    if not results['changed_cases']:
        print("  No cases found.")
        return
    
    for i, case in enumerate(results['changed_cases'][:max_cases]):
        player = case['player_cards']
        dealer = case['dealer_card']
        shoe_counts = case.get('shoe_counts', {})
        cards_left = sum(shoe_counts.values()) if shoe_counts else 'N/A'
        
        print(f"\nCase {i+1}: Player {player} vs Dealer {dealer}")
        print(f"  Cards remaining: {cards_left}")
        print(f"  Shoe: {shoe_counts}")
        print(f"  First action changed: {case['initial_first_action']} -> {case['final_first_action']}")
        
        print("  Initial best actions (action, ev, ev_min, ev_max):")
        for action, ev, ev_min, ev_max in case['initial_best_actions']:
            gap = ev_max - ev_min
            print(f"    {action:20s}: EV={ev:10.6f}  [{ev_min:10.6f}, {ev_max:10.6f}]  gap={gap:.6f}")
        
        print("  Final best actions (after tightening):")
        for action, ev, ev_min, ev_max in case['final_best_actions']:
            gap = ev_max - ev_min
            print(f"    {action:20s}: EV={ev:10.6f}  [{ev_min:10.6f}, {ev_max:10.6f}]  gap={gap:.6f}")

print("=" * 100)
print("CHANGED CASES SUMMARY")
print("=" * 100)

if results_1e6['changed_cases']:
    show_changed_cases(results_1e6, max_cases=10)
else:
    print("\nNo cases where first action changed with gap=1e-6")


CHANGED CASES SUMMARY

No cases where first action changed with gap=1e-6


## Summary

### Investigation 1: get_best_action() returning None

When the TreeWalker cannot conclusively determine a single best action (because multiple actions have overlapping floor/ceiling value ranges), `get_best_action()` now returns `None` instead of throwing an exception.

This allows the caller to decide how to handle ambiguous situations:
- Use `get_best_actions()[0]` to pick the action with highest current EV estimate
- Call `tighten_value_estimate_gap()` to refine estimates
- Use a fallback heuristic

### Investigation 2: Effect of tighten_value_estimate_gap

This tests whether tightening the gap can change which action appears best, either:
- Changing `get_best_action()` from None to a definitive answer (expected behavior)
- Changing which action is first in `get_best_actions()` (would indicate EV ordering flipped)


## Analysis of Ambiguous Cases

Let's analyze the ambiguous cases found during the exhaustive shoe simulation to understand:
- At what shoe penetration do ambiguous decisions occur?
- Which hand types most commonly have ambiguity?
- What action pairs compete for best?


In [9]:
# Analyze ambiguity by cards remaining in shoe
# This shows at what shoe penetration ambiguous decisions occur

if results['ambiguous_cases']:
    cards_remaining_list = [case['cards_remaining'] for case in results['ambiguous_cases']]
    
    print("Ambiguity analysis by shoe penetration:")
    print("=" * 60)
    print(f"Total ambiguous cases: {len(cards_remaining_list)}")
    print(f"Min cards remaining: {min(cards_remaining_list)}")
    print(f"Max cards remaining: {max(cards_remaining_list)}")
    print(f"Mean cards remaining: {np.mean(cards_remaining_list):.1f}")
    print(f"Median cards remaining: {np.median(cards_remaining_list):.1f}")
    
    # Histogram of cards remaining
    bins = [20, 50, 100, 150, 200, 250, 312]
    hist, _ = np.histogram(cards_remaining_list, bins=bins)
    print(f"\nDistribution by cards remaining:")
    for i in range(len(bins) - 1):
        print(f"  {bins[i]:3d}-{bins[i+1]:3d} cards: {hist[i]:5d} cases")
else:
    print("No ambiguous cases found to analyze.")


No ambiguous cases found to analyze.


In [10]:
# Analyze which hand types most commonly have ambiguity
if results['ambiguous_cases']:
    from collections import Counter
    
    # Count by player hand total and dealer upcard
    hand_totals = []
    dealer_upcards = []
    competing_actions = []
    
    for case in results['ambiguous_cases']:
        p1, p2 = case['player_cards'][:2]  # First two cards
        total = p1 + p2
        if p1 == 11 or p2 == 11:  # Soft hand
            hand_totals.append(f"Soft {total}")
        elif p1 == p2:  # Pair
            hand_totals.append(f"Pair {p1}")
        else:
            hand_totals.append(f"Hard {total}")
        
        dealer_upcards.append(case['dealer_card'])
        
        # Get the competing actions
        actions = [a[0] for a in case['best_actions'][:2]]
        competing_actions.append(tuple(sorted(actions)))
    
    print("Most common ambiguous hand types:")
    print("=" * 60)
    for hand, count in Counter(hand_totals).most_common(15):
        print(f"  {hand:15s}: {count:5d} cases")
    
    print("\nMost common competing action pairs:")
    print("=" * 60)
    for actions, count in Counter(competing_actions).most_common(10):
        print(f"  {actions}: {count:5d} cases")
    
    print("\nAmbiguity by dealer upcard:")
    print("=" * 60)
    for upcard, count in sorted(Counter(dealer_upcards).items()):
        upcard_str = "A" if upcard == 11 else str(upcard)
        print(f"  Dealer {upcard_str:2s}: {count:5d} cases")
else:
    print("No ambiguous cases to analyze.")


No ambiguous cases to analyze.
